<div style="display: flex; background-color: DarkBlue;">
<div style="margin: auto; padding: 48px; color: white; font-size: 48px; font-weight: bold" align="center">
DWFA – Drinking Water For All
<br/>
💧 Étude sur l'eau potable 🚰
</div>
</div>

<div style="font-size: 42px;">
📓 Notebook préliminaire : analyse, nettoyage et transformation des données brutes
</div>

# Imports et paramètres généraux

In [1]:
import pandas as pd

In [2]:
from pathlib import Path

In [3]:
data_path = Path("../data")

In [4]:
from functools import reduce

# Chargement des fichiers

## Services (basique, qualité)

In [5]:
df_services = pd.read_csv(data_path/"raw"/"BasicAndSafelyManagedDrinkingWaterServices.csv")
df_services

,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
0,2000,Afghanistan,Rural,21.61913,NaN
1,2000,Afghanistan,Total,27.77190,NaN
2,2000,Afghanistan,Urban,49.48745,NaN
3,2000,Albania,Rural,81.78472,NaN
4,2000,Albania,Total,87.86662,49.29324
...,...,...,...,...,...
10471,2017,Zambia,Total,59.96376,NaN
10472,2017,Zambia,Urban,83.86312,46.24515
10473,2017,Zimbabwe,Rural,49.80476,NaN
10474,2017,Zimbabwe,Total,64.05123,NaN


In [6]:
df_services.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 10476 entries, 0 to 10475
Data columns (total 5 columns):
 #   Column                                                       Non-Null Count  Dtype  
---  ------                                                       --------------  -----  
 0   Year                                                         10476 non-null  int64  
 1   Country                                                      10476 non-null  str    
 2   Granularity                                                  10476 non-null  str    
 3   Population using at least basic drinking-water services (%)  9415 non-null   float64
 4   Population using safely managed drinking-water services (%)  3286 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 562.0 KB


In [7]:
df_services["Year"].value_counts()

Year
2000    582
2001    582
2002    582
2003    582
2004    582
2005    582
2006    582
2007    582
2008    582
2009    582
2010    582
2011    582
2012    582
2013    582
2014    582
2015    582
2016    582
2017    582
Name: count, dtype: int64

In [8]:
df_services["Country"].value_counts()

Country
Afghanistan                           54
Albania                               54
Algeria                               54
Andorra                               54
Angola                                54
                                      ..
Venezuela (Bolivarian Republic of)    54
Viet Nam                              54
Yemen                                 54
Zambia                                54
Zimbabwe                              54
Name: count, Length: 194, dtype: int64

In [9]:
df_services["Granularity"].value_counts()

Granularity
Rural    3492
Total    3492
Urban    3492
Name: count, dtype: int64

In [10]:
df_services.iloc[:,3:].describe().T

,count,mean,std,min,25%,50%,75%,max
Population using at least basic drinking-water services (%),9415.0,83.962120,19.968269,4.08262,75.928395,93.115400,98.95424,100.00001
Population using safely managed drinking-water services (%),3286.0,66.070856,30.383942,0.00000,41.895583,73.966655,94.77664,100.00000


In [11]:
df_services[ (df_services.iloc[:,3:].max(axis=1) > 100) ]

,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
232,2000,Iceland,Total,100.00001,89.53967
3142,2005,Iceland,Total,100.00001,92.67429
3502,2006,Andorra,Total,100.00001,90.64001
5422,2009,Finland,Total,100.00001,97.81097
7168,2012,Finland,Total,100.00001,99.58952
9703,2016,Palau,Total,100.00001,NaN
10276,2017,Norway,Total,100.00001,98.34297


In [12]:
for column in list(df_services.columns[3:]):
    df_services[column] = (df_services[column] / 100).clip(0, 1)
    df_services.rename(columns={column:column[:-4]}, inplace=True)
df_services.iloc[:,3:].describe().T

,count,mean,std,min,25%,50%,75%,max
Population using at least basic drinking-water services,9415.0,0.839621,0.199683,0.040826,0.759284,0.931154,0.989542,1.0
Population using safely managed drinking-water services,3286.0,0.660709,0.303839,0.000000,0.418956,0.739667,0.947766,1.0


## Mortalité (nombre de personnes, ratio pour 100k)

In [13]:
df_mortality = pd.read_csv(data_path/"raw"/"MortalityRateAttributedToWater.csv")
df_mortality

,Year,Country,Granularity,Mortality rate attributed to exposure to unsafe WASH services,WASH deaths
0,2016,Afghanistan,Female,15.31193,NaN
1,2016,Afghanistan,Male,12.61297,NaN
2,2016,Afghanistan,Total,13.92067,4824.353
3,2016,Albania,Female,0.12552,NaN
4,2016,Albania,Male,0.20650,NaN
...,...,...,...,...,...
544,2016,Zambia,Male,36.61913,NaN
545,2016,Zambia,Total,34.91273,5792.504
546,2016,Zimbabwe,Female,22.16388,NaN
547,2016,Zimbabwe,Male,27.06688,NaN


In [14]:
df_mortality.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 549 entries, 0 to 548
Data columns (total 5 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   Year                                                           549 non-null    int64  
 1   Country                                                        549 non-null    str    
 2   Granularity                                                    549 non-null    str    
 3   Mortality rate attributed to exposure to unsafe WASH services  549 non-null    float64
 4   WASH deaths                                                    183 non-null    float64
dtypes: float64(2), int64(1), str(2)
memory usage: 29.6 KB


In [15]:
df_mortality["Year"].value_counts()

Year
2016    549
Name: count, dtype: int64

In [16]:
df_mortality["Country"].value_counts()

Country
Afghanistan                           3
Albania                               3
Algeria                               3
Angola                                3
Antigua and Barbuda                   3
                                     ..
Venezuela (Bolivarian Republic of)    3
Viet Nam                              3
Yemen                                 3
Zambia                                3
Zimbabwe                              3
Name: count, Length: 183, dtype: int64

In [17]:
df_mortality["Granularity"].value_counts()

Granularity
Female    183
Male      183
Total     183
Name: count, dtype: int64

In [18]:
df_mortality.iloc[:,3:].describe().T

,count,mean,std,min,25%,50%,75%,max
Mortality rate attributed to exposure to unsafe WASH services,549.0,12.493876,20.830508,0.00396,0.192960,1.28871,18.05478,107.04802
WASH deaths,183.0,4756.097706,21280.125369,0.08229,11.163275,130.98340,1950.43350,246087.90000


## Stabilité politique

In [19]:
df_stability = pd.read_csv(data_path/"raw"/"PoliticalStability.csv")
df_stability

,Country,Year,Political_Stability,Granularity
0,Afghanistan,2000,-2.44,Total
1,Afghanistan,2002,-2.04,Total
2,Afghanistan,2003,-2.20,Total
3,Afghanistan,2004,-2.30,Total
4,Afghanistan,2005,-2.07,Total
...,...,...,...,...
3521,Zimbabwe,2014,-0.71,Total
3522,Zimbabwe,2015,-0.62,Total
3523,Zimbabwe,2016,-0.62,Total
3524,Zimbabwe,2017,-0.71,Total


In [20]:
df_stability["Political stability"] = df_stability.pop("Political_Stability") # Renommer et mettre en dernier
df_stability.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 3526 entries, 0 to 3525
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              3526 non-null   str    
 1   Year                 3526 non-null   int64  
 2   Granularity          3526 non-null   str    
 3   Political stability  3526 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 162.3 KB


In [21]:
df_stability["Year"].value_counts()

Year
2009    199
2010    199
2011    198
2012    198
2013    198
2014    198
2015    198
2016    198
2017    198
2018    198
2006    196
2007    196
2008    196
2004    195
2005    195
2003    194
2000    186
2002    186
Name: count, dtype: int64

In [22]:
df_stability["Country"].value_counts()

Country
Afghanistan     18
Albania         18
Algeria         18
Andorra         18
Angola          18
                ..
Montenegro      13
Greenland       10
South Sudan      8
Cook Islands     2
Niue             2
Name: count, Length: 200, dtype: int64

In [23]:
df_stability["Granularity"].value_counts()

Granularity
Total    3526
Name: count, dtype: int64

In [24]:
df_stability.iloc[:,3:].describe().T

,count,mean,std,min,25%,50%,75%,max
Political stability,3526.0,-0.051044,0.996039,-3.31,-0.71,0.05,0.7975,1.97


## Population

In [25]:
df_population = pd.read_csv(data_path/"raw"/"Population.csv")
df_population

,Country,Granularity,Year,Population
0,Afghanistan,Total,2000,20779.953
1,Afghanistan,Male,2000,10689.508
2,Afghanistan,Female,2000,10090.449
3,Afghanistan,Rural,2000,15657.474
4,Afghanistan,Urban,2000,4436.282
...,...,...,...,...
20909,Zimbabwe,Total,2018,14438.802
20910,Zimbabwe,Male,2018,6879.119
20911,Zimbabwe,Female,2018,7559.693
20912,Zimbabwe,Rural,2018,11465.748


In [26]:
df_population.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 20914 entries, 0 to 20913
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      20914 non-null  str    
 1   Granularity  20914 non-null  str    
 2   Year         20914 non-null  int64  
 3   Population   20914 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 973.9 KB


In [27]:
df_population["Year"].value_counts()

Year
2012    1113
2013    1113
2014    1113
2015    1113
2016    1113
2017    1113
2018    1113
2011    1108
2006    1095
2007    1095
2008    1095
2009    1095
2010    1095
2000    1090
2001    1090
2002    1090
2003    1090
2004    1090
2005    1090
Name: count, dtype: int64

In [28]:
df_population["Country"].value_counts()

Country
Afghanistan                         95
Albania                             95
Algeria                             95
Angola                              95
Antigua and Barbuda                 95
                                    ..
Serbia and Montenegro               30
Bonaire, Sint Eustatius and Saba    24
Sint Maarten  (Dutch part)          24
Saint Barthélemy                     8
Saint-Martin (French part)           8
Name: count, Length: 239, dtype: int64

In [29]:
df_population["Granularity"].value_counts()

Granularity
Total     4430
Rural     4414
Urban     4414
Male      3828
Female    3828
Name: count, dtype: int64

In [30]:
df_population.iloc[:,3:].describe().T

,count,mean,std,min,25%,50%,75%,max
Population,20914.0,22531.640692,100016.850913,0.0,348.34625,3016.3365,11150.426,1459377.612


In [31]:
df_population["Population"] *= 1000
df_population.iloc[:,3:].describe().T

,count,mean,std,min,25%,50%,75%,max
Population,20914.0,2.253164e+07,1.000169e+08,0.0,348346.25,3016336.5,11150426.0,1.459378e+09


## Regroupements de pays

In [32]:
df_region_country = pd.read_csv(data_path/"raw"/"RegionCountry.csv")
df_region_country

,REGION (DISPLAY),COUNTRY (DISPLAY)
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia
3,Western Pacific,Australia
4,Europe,Austria
...,...,...
189,Eastern Mediterranean,United Arab Emirates
190,Americas,Venezuela (Bolivarian Republic of)
191,Western Pacific,Viet Nam
192,Eastern Mediterranean,Yemen


In [33]:
df_region_country.columns = ["Region","Country"]
df_region_country.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Region   194 non-null    str  
 1   Country  194 non-null    str  
dtypes: str(2)
memory usage: 6.9 KB


In [34]:
df_region_country["Region"].value_counts()

Region
Europe                   53
Africa                   47
Americas                 35
Western Pacific          27
Eastern Mediterranean    21
South-East Asia          11
Name: count, dtype: int64

In [35]:
df_region_country["Country"].value_counts()

Country
Albania                               1
Andorra                               1
Armenia                               1
Australia                             1
Austria                               1
                                     ..
United Arab Emirates                  1
Venezuela (Bolivarian Republic of)    1
Viet Nam                              1
Yemen                                 1
Zimbabwe                              1
Name: count, Length: 194, dtype: int64

In [36]:
(df_region_country["Country"].value_counts() == 1).all()

np.True_

# Analyse croisée : `Country`

In [37]:
dfs = {
    "services": df_services,
    "mortality": df_mortality,
    "stability": df_stability,
    "population": df_population,
    "region_country": df_region_country,
}

In [38]:
for key, df in dfs.items():
    print(f"{key}: {len(df["Country"].unique())} countries")

services: 194 countries
mortality: 183 countries
stability: 200 countries
population: 239 countries
region_country: 194 countries


## Pays en commun et/ou spécifiques

### Union

In [39]:
countries_all = reduce(pd.Index.union, (pd.Index(df["Country"].unique()) for df in dfs.values()))
print(f"Total = {len(countries_all)} countries")
print(*countries_all, sep="\n")

Total = 240 countries
Afghanistan
Albania
Algeria
American Samoa
Andorra
Angola
Anguilla
Antigua and Barbuda
Argentina
Armenia
Aruba
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bermuda
Bhutan
Bolivia (Plurinational State of)
Bonaire, Sint Eustatius and Saba
Bosnia and Herzegovina
Botswana
Brazil
British Virgin Islands
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Cayman Islands
Central African Republic
Chad
Channel Islands
Chile
China
China, Hong Kong SAR
China, Macao SAR
China, Taiwan Province of
China, mainland
Colombia
Comoros
Congo
Cook Islands
Costa Rica
Croatia
Cuba
Curaçao
Cyprus
Czechia
Côte d'Ivoire
Democratic People's Republic of Korea
Democratic Republic of the Congo
Denmark
Djibouti
Dominica
Dominican Republic
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Falkland Islands (Malvinas)
Faroe Islands
Fiji
Finland
France
French Guyana
French Polynesia
Gabon
Gam

### Intersection

In [40]:
countries_common = reduce(pd.Index.intersection, (pd.Index(df["Country"].unique()) for df in dfs.values()))
print(f"Total = {len(countries_common)} countries in common")
print(*countries_common, sep="\n")

Total = 181 countries in common
Afghanistan
Albania
Algeria
Angola
Antigua and Barbuda
Argentina
Armenia
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bhutan
Bolivia (Plurinational State of)
Bosnia and Herzegovina
Botswana
Brazil
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Central African Republic
Chad
Chile
Colombia
Comoros
Congo
Costa Rica
Croatia
Cuba
Cyprus
Czechia
Côte d'Ivoire
Democratic People's Republic of Korea
Democratic Republic of the Congo
Denmark
Djibouti
Dominican Republic
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Fiji
Finland
France
Gabon
Gambia
Georgia
Germany
Ghana
Greece
Grenada
Guatemala
Guinea
Guinea-Bissau
Guyana
Haiti
Honduras
Hungary
Iceland
India
Indonesia
Iran (Islamic Republic of)
Iraq
Ireland
Israel
Italy
Jamaica
Japan
Jordan
Kazakhstan
Kenya
Kiribati
Kuwait
Kyrgyzstan
Lao People's Democratic Republic
Latvia
Lebanon
Lesotho
Liberia
Liby

### Différence

In [41]:
countries_diff = countries_all.difference(countries_common)
print(f"Total = {len(countries_diff)} specific countries")
print(*countries_diff, sep="\n")

Total = 59 specific countries
American Samoa
Andorra
Anguilla
Aruba
Bermuda
Bonaire, Sint Eustatius and Saba
British Virgin Islands
Cayman Islands
Channel Islands
China
China, Hong Kong SAR
China, Macao SAR
China, Taiwan Province of
China, mainland
Cook Islands
Curaçao
Dominica
Falkland Islands (Malvinas)
Faroe Islands
French Guyana
French Polynesia
Gibraltar
Greenland
Guadeloupe
Guam
Holy See
Isle of Man
Liechtenstein
Marshall Islands
Martinique
Mayotte
Monaco
Montserrat
Nauru
Netherlands Antilles (former)
New Caledonia
Niue
North Macedonia
Northern Mariana Islands
Palau
Palestine
Puerto Rico
Republic of North Macedonia
Réunion
Saint Barthélemy
Saint Helena, Ascension and Tristan da Cunha
Saint Kitts and Nevis
Saint Pierre and Miquelon
Saint-Martin (French part)
San Marino
Serbia and Montenegro
Sint Maarten  (Dutch part)
Sudan (former)
Tokelau
Turks and Caicos Islands
Tuvalu
United States Virgin Islands
Wallis and Futuna Islands
Western Sahara


## Ancien (former)

In [42]:
countries_former = countries_all[ countries_all.str.contains("former") ]
print(f"Total = {len(countries_former)} former countries")
print(*countries_former, sep="\n")

Total = 2 former countries
Netherlands Antilles (former)
Sudan (former)


## ISO3

Le fichier **iso3.csv** a été généré par Mistral (agent IA) à partir de la liste de tous les noms de pays.

In [43]:
df_iso3 = pd.read_csv(data_path/"ia"/"iso3.csv", index_col="Country")
df_iso3

,ISO3
Country,
Afghanistan,AFG
Albania,ALB
Algeria,DZA
American Samoa,ASM
Andorra,AND
...,...
Wallis and Futuna Islands,WLF
Western Sahara,ESH
Yemen,YEM


In [44]:
assert (countries_all == df_iso3.index).all()

In [45]:
df_iso3_repeat = df_iso3.groupby("ISO3").size()
df_iso3_repeat = df_iso3_repeat[ df_iso3_repeat != 1 ]
df_iso3_repeat

ISO3
CHN    2
MKD    2
SDN    2
dtype: int64

In [46]:
print()
for iso3 in df_iso3_repeat.index:
    print(iso3, *df_iso3[ df_iso3["ISO3"] == iso3 ].index, sep="\n\t")
    print()


CHN
	China
	China, mainland

MKD
	North Macedonia
	Republic of North Macedonia

SDN
	Sudan
	Sudan (former)



## Utilisation des noms de pays par fichier

In [47]:
df_country_file = pd.DataFrame(index=countries_all)
for key, df in dfs.items():
    df_country_file[key] = df_country_file.index.isin(df["Country"])
df_country_file

,services,mortality,stability,population,region_country
Afghanistan,True,True,True,True,True
Albania,True,True,True,True,True
Algeria,True,True,True,True,True
American Samoa,False,False,True,True,False
Andorra,True,False,True,True,True
...,...,...,...,...,...
Wallis and Futuna Islands,False,False,False,True,False
Western Sahara,False,False,False,True,False
Yemen,True,True,True,True,True
Zambia,True,True,True,True,True


In [48]:
df_country_file[ ~df_country_file["population"] ]

,services,mortality,stability,population,region_country
Republic of North Macedonia,True,True,False,False,True


In [49]:
df_country_file[ (df_country_file.sum(axis=1) == 1) & df_country_file["population"] ]

,services,mortality,stability,population,region_country
Anguilla,False,False,False,True,False
Aruba,False,False,False,True,False
"Bonaire, Sint Eustatius and Saba",False,False,False,True,False
British Virgin Islands,False,False,False,True,False
Cayman Islands,False,False,False,True,False
Channel Islands,False,False,False,True,False
Curaçao,False,False,False,True,False
Falkland Islands (Malvinas),False,False,False,True,False
Faroe Islands,False,False,False,True,False
French Guyana,False,False,False,True,False


In [50]:
df_country_file[ (df_country_file.sum(axis=1) == 2) & df_country_file["population"] ]

,services,mortality,stability,population,region_country
American Samoa,False,False,True,True,False
Bermuda,False,False,True,True,False
"China, Hong Kong SAR",False,False,True,True,False
"China, Macao SAR",False,False,True,True,False
"China, Taiwan Province of",False,False,True,True,False
"China, mainland",False,False,True,True,False
Greenland,False,False,True,True,False
North Macedonia,False,False,True,True,False
Palestine,False,False,True,True,False
Puerto Rico,False,False,True,True,False


In [51]:
df_country_file[ (df_country_file.sum(axis=1) == 3) & df_country_file["population"] ]

,services,mortality,stability,population,region_country
Monaco,True,False,False,True,True
San Marino,True,False,False,True,True


In [52]:
df_country_file[ (df_country_file.sum(axis=1) == 4) & df_country_file["population"] ]

,services,mortality,stability,population,region_country
Andorra,True,False,True,True,True
China,True,True,False,True,True
Cook Islands,True,False,True,True,True
Dominica,True,False,True,True,True
Marshall Islands,True,False,True,True,True
Nauru,True,False,True,True,True
Niue,True,False,True,True,True
Palau,True,False,True,True,True
Saint Kitts and Nevis,True,False,True,True,True
Tuvalu,True,False,True,True,True


## Identification des libellés inclus dans d'autres libellés

In [53]:
# pd.describe_option()
pd.set_option("display.width", 200)
print()
for country in df_country_file.index:
    index = df_country_file.index[ df_country_file.index.str.contains(country, case=False, regex=False) ]
    if len(index) >= 2:
        print(df_country_file.loc[index])
        print()
pd.reset_option("display.width")


                           services  mortality  stability  population  region_country
China                          True       True      False        True            True
China, Hong Kong SAR          False      False       True        True           False
China, Macao SAR              False      False       True        True           False
China, Taiwan Province of     False      False       True        True           False
China, mainland               False      False       True        True           False

                                  services  mortality  stability  population  region_country
Congo                                 True       True       True        True            True
Democratic Republic of the Congo      True       True       True        True            True

                    services  mortality  stability  population  region_country
Dominica                True      False       True        True            True
Dominican Republic      True       True     

# Cas particulier : Chine

## Population

In [54]:
df_pop_china = df_population[ df_population["Country"].str.contains("China", case=False) ].copy()
df_pop_china

,Country,Granularity,Year,Population
3786,China,Total,2000,1.319551e+09
3787,China,Male,2000,6.768010e+08
3788,China,Female,2000,6.427504e+08
3789,China,Rural,2000,8.294033e+08
3790,China,Urban,2000,4.827275e+08
...,...,...,...,...
4256,"China, Taiwan Province of",Total,2018,2.372646e+07
4257,"China, Taiwan Province of",Male,2018,1.181217e+07
4258,"China, Taiwan Province of",Female,2018,1.191429e+07
4259,"China, Taiwan Province of",Rural,2018,5.175798e+06


In [55]:
df_pop_china["Country"].value_counts()

Country
China                        95
China, Hong Kong SAR         95
China, Macao SAR             95
China, mainland              95
China, Taiwan Province of    95
Name: count, dtype: int64

In [56]:
df_pop_china.loc[(df_pop_china["Country"] == "China"),"Population"] *= -1
df_pop_china

,Country,Granularity,Year,Population
3786,China,Total,2000,-1.319551e+09
3787,China,Male,2000,-6.768010e+08
3788,China,Female,2000,-6.427504e+08
3789,China,Rural,2000,-8.294033e+08
3790,China,Urban,2000,-4.827275e+08
...,...,...,...,...
4256,"China, Taiwan Province of",Total,2018,2.372646e+07
4257,"China, Taiwan Province of",Male,2018,1.181217e+07
4258,"China, Taiwan Province of",Female,2018,1.191429e+07
4259,"China, Taiwan Province of",Rural,2018,5.175798e+06


In [57]:
max_abs_diff = df_pop_china.groupby(["Granularity","Year"])["Population"].sum().abs().max()
print(f"China = mainland + Hong Kong + Macao + Taiwan ? {'TOUJOURS' if max_abs_diff < 1e-6 else 'PARFOIS NON'}")

China = mainland + Hong Kong + Macao + Taiwan ? TOUJOURS


In [58]:
df_country_file.loc[df_pop_china["Country"].unique()]

,services,mortality,stability,population,region_country
China,True,True,False,True,True
"China, Hong Kong SAR",False,False,True,True,False
"China, Macao SAR",False,False,True,True,False
"China, mainland",False,False,True,True,False
"China, Taiwan Province of",False,False,True,True,False


## Stabilité politique

In [59]:
print()
years = df_stability["Year"].unique()
years.sort()
for year in years:
    print(df_stability[ (df_stability["Year"] == year) & df_stability["Country"].str.contains("China") ])
    print()


                       Country  Year Granularity  Political stability
663       China, Hong Kong SAR  2000       Total                 0.93
681           China, Macao SAR  2000       Total                 0.49
699            China, mainland  2000       Total                -0.21
717  China, Taiwan Province of  2000       Total                 0.54

                       Country  Year Granularity  Political stability
664       China, Hong Kong SAR  2002       Total                 0.91
682           China, Macao SAR  2002       Total                 0.55
700            China, mainland  2002       Total                -0.33
718  China, Taiwan Province of  2002       Total                 0.70

                       Country  Year Granularity  Political stability
665       China, Hong Kong SAR  2003       Total                 0.95
683           China, Macao SAR  2003       Total                 1.12
701            China, mainland  2003       Total                -0.56
719  China, Taiwa

# Résumé des observations, hypothèses et choix techniques

À ce stade, nous avons observé que les libellés pour un même territoire (parfois plus petit qu'un pays) peuvent varier d'un fichier à l'autre.
Certains territoires ne sont présents que dans certains fichiers, d'autres dans tous.

Voici un résumé des observations ainsi que les hypothèses émises afin d'uniformiser les libellés.
Cela guidera la suite du travail préparatoire et permettra de charger directement des données cohérentes et prêtes-à-l'emploi dans le tableau de bord.

## Chine

Les deux fichiers issus de la FAO (population, stabilité politique) ont un traitement spécial pour la Chine.

- Informations pour 4 parties (Chine continentale, Hong Kong, Macao, Taiwan) : nous les conservons.
- Informations agrégées pour l'ensemble (population uniquement) : nous les excluons.

Ainsi, il n'y a plus d'information redondante, ni de risque de surestimer la population chinoise ou mondiale.

De plus, nous supposons que, dans les autres fichiers, le libellé "China" désigne la Chine continentale.

## Macédoine du Nord

Les fichiers issus de la FAO utilisent le libellé "North Macedonia".
Les autres fichiers utilisent le libellé "Republic of North Macedonia" : nous conservons ce dernier.

## Soudan

Le Soudan du Sud s'est désolidarisé du Soudan en 2011.
Cela explique pourquoi il y a deux libellés pour le Soudan (avec ou sans le suffixe "(former)").
Nous conservons les 3 libellés qui permettent de distinguer les informations avant/après cette séparation.

# Uniformisation des noms des pays

## Transformations

In [60]:
mapping_fao = {
    "China": None,
    "China, mainland": "China",
    "China, Hong Kong SAR": "Hong Kong",
    "China, Macao SAR": "Macao",
    "China, Taiwan Province of": "Taiwan",
    "North Macedonia": "Republic of North Macedonia",
}
mapping_fao

{'China': None,
 'China, mainland': 'China',
 'China, Hong Kong SAR': 'Hong Kong',
 'China, Macao SAR': 'Macao',
 'China, Taiwan Province of': 'Taiwan',
 'North Macedonia': 'Republic of North Macedonia'}

In [61]:
pd.DataFrame(mapping_fao.items(), columns=["Before","After"])

,Before,After
0,China,NaN
1,"China, mainland",China
2,"China, Hong Kong SAR",Hong Kong
3,"China, Macao SAR",Macao
4,"China, Taiwan Province of",Taiwan
5,North Macedonia,Republic of North Macedonia


In [62]:
df_population["Country"] = df_population["Country"].map(lambda before: mapping_fao.get(before, before))
df_population = df_population.dropna()

In [63]:
df_stability["Country"] = df_stability["Country"].map(lambda before: mapping_fao.get(before, before))
df_stability = df_stability.dropna()

In [64]:
df_iso3 = df_iso3.reset_index()
df_iso3["Country"] = df_iso3["Country"].map(lambda before: mapping_fao.get(before, before))
df_iso3 = df_iso3.dropna()
df_iso3 = df_iso3.drop_duplicates()
df_iso3

,Country,ISO3
0,Afghanistan,AFG
1,Albania,ALB
2,Algeria,DZA
3,American Samoa,ASM
4,Andorra,AND
...,...,...
235,Wallis and Futuna Islands,WLF
236,Western Sahara,ESH
237,Yemen,YEM
238,Zambia,ZMB


In [65]:
df_iso3_repeat = df_iso3.groupby("ISO3").size()
df_iso3_repeat = df_iso3_repeat[ df_iso3_repeat != 1 ]
df_iso3_repeat

ISO3
SDN    2
dtype: int64

## Vérifications

### Union

In [66]:
countries_all = reduce(pd.Index.union, (pd.Index(df["Country"].unique()) for df in dfs.values()))
print(f"Total = {len(countries_all)} countries")
print(*countries_all, sep="\n")

Total = 239 countries
Afghanistan
Albania
Algeria
American Samoa
Andorra
Angola
Anguilla
Antigua and Barbuda
Argentina
Armenia
Aruba
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bermuda
Bhutan
Bolivia (Plurinational State of)
Bonaire, Sint Eustatius and Saba
Bosnia and Herzegovina
Botswana
Brazil
British Virgin Islands
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Cayman Islands
Central African Republic
Chad
Channel Islands
Chile
China
Colombia
Comoros
Congo
Cook Islands
Costa Rica
Croatia
Cuba
Curaçao
Cyprus
Czechia
Côte d'Ivoire
Democratic People's Republic of Korea
Democratic Republic of the Congo
Denmark
Djibouti
Dominica
Dominican Republic
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Falkland Islands (Malvinas)
Faroe Islands
Fiji
Finland
France
French Guyana
French Polynesia
Gabon
Gambia
Georgia
Germany
Ghana
Gibraltar
Greece
Greenland
Grenada
Guadeloupe
Guam
Gua

### Intersection

In [67]:
countries_common = reduce(pd.Index.intersection, (pd.Index(df["Country"].unique()) for df in dfs.values()))
print(f"Total = {len(countries_common)} countries in common")
print(*countries_common, sep="\n")

Total = 183 countries in common
Afghanistan
Albania
Algeria
Angola
Antigua and Barbuda
Argentina
Armenia
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bhutan
Bolivia (Plurinational State of)
Bosnia and Herzegovina
Botswana
Brazil
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Central African Republic
Chad
Chile
China
Colombia
Comoros
Congo
Costa Rica
Croatia
Cuba
Cyprus
Czechia
Côte d'Ivoire
Democratic People's Republic of Korea
Democratic Republic of the Congo
Denmark
Djibouti
Dominican Republic
Ecuador
Egypt
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Fiji
Finland
France
Gabon
Gambia
Georgia
Germany
Ghana
Greece
Grenada
Guatemala
Guinea
Guinea-Bissau
Guyana
Haiti
Honduras
Hungary
Iceland
India
Indonesia
Iran (Islamic Republic of)
Iraq
Ireland
Israel
Italy
Jamaica
Japan
Jordan
Kazakhstan
Kenya
Kiribati
Kuwait
Kyrgyzstan
Lao People's Democratic Republic
Latvia
Lebanon
Lesotho
Liberi

### Différence

In [68]:
countries_diff = countries_all.difference(countries_common)
print(f"Total = {len(countries_diff)} specific countries")
print(*countries_diff, sep="\n")

Total = 56 specific countries
American Samoa
Andorra
Anguilla
Aruba
Bermuda
Bonaire, Sint Eustatius and Saba
British Virgin Islands
Cayman Islands
Channel Islands
Cook Islands
Curaçao
Dominica
Falkland Islands (Malvinas)
Faroe Islands
French Guyana
French Polynesia
Gibraltar
Greenland
Guadeloupe
Guam
Holy See
Hong Kong
Isle of Man
Liechtenstein
Macao
Marshall Islands
Martinique
Mayotte
Monaco
Montserrat
Nauru
Netherlands Antilles (former)
New Caledonia
Niue
Northern Mariana Islands
Palau
Palestine
Puerto Rico
Réunion
Saint Barthélemy
Saint Helena, Ascension and Tristan da Cunha
Saint Kitts and Nevis
Saint Pierre and Miquelon
Saint-Martin (French part)
San Marino
Serbia and Montenegro
Sint Maarten  (Dutch part)
Sudan (former)
Taiwan
Tokelau
Turks and Caicos Islands
Tuvalu
United States Virgin Islands
Wallis and Futuna Islands
Western Sahara
nan


## Mise à jour des regroupements régionaux

In [69]:
china_region = df_region_country["Region"][ (df_region_country["Country"] == "China") ].iloc[0]
china_region

'Western Pacific'

In [70]:
df_region_country_china_parts = pd.DataFrame([
    [ china_region, val ]
    for key, val in mapping_fao.items()
    if key.startswith("China, ") and val != "China"
], columns=["Region","Country"])
df_region_country_china_parts

,Region,Country
0,Western Pacific,Hong Kong
1,Western Pacific,Macao
2,Western Pacific,Taiwan


In [71]:
df_region_country = pd.concat([ df_region_country, df_region_country_china_parts ], ignore_index=True)
df_region_country

,Region,Country
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia
3,Western Pacific,Australia
4,Europe,Austria
...,...,...
192,Eastern Mediterranean,Yemen
193,Africa,Zimbabwe
194,Western Pacific,Hong Kong
195,Western Pacific,Macao


# Construction de la table de faits (format long)

In [72]:
del dfs["region_country"] # Données propres à la dimension Country (ni année ni granularité, seulement rattachement régional)

## Dépivotage

In [73]:
print()
for key, df in dfs.items():
    print(key)
    df = df.melt(id_vars=["Country","Year","Granularity"], var_name="Indicator", value_name="Value")
    print(df.info(verbose=True))
    dfs[key] = df
    print()


services
<class 'pandas.DataFrame'>
RangeIndex: 20952 entries, 0 to 20951
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      20952 non-null  str    
 1   Year         20952 non-null  int64  
 2   Granularity  20952 non-null  str    
 3   Indicator    20952 non-null  str    
 4   Value        12701 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 2.2 MB
None

mortality
<class 'pandas.DataFrame'>
RangeIndex: 1098 entries, 0 to 1097
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      1098 non-null   str    
 1   Year         1098 non-null   int64  
 2   Granularity  1098 non-null   str    
 3   Indicator    1098 non-null   str    
 4   Value        732 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 97.8 KB
None

stability
<class 'pandas.DataFrame'>
RangeIndex: 3526 entries, 0 to 352

## Suppression des valeurs manquantes

In [74]:
print()
for key, df in dfs.items():
    print(key)
    df = df.dropna()
    print(df.info(verbose=True))
    dfs[key] = df
    print()


services
<class 'pandas.DataFrame'>
Index: 12701 entries, 0 to 20948
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      12701 non-null  str    
 1   Year         12701 non-null  int64  
 2   Granularity  12701 non-null  str    
 3   Indicator    12701 non-null  str    
 4   Value        12701 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 1.4 MB
None

mortality
<class 'pandas.DataFrame'>
Index: 732 entries, 0 to 1097
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      732 non-null    str    
 1   Year         732 non-null    int64  
 2   Granularity  732 non-null    str    
 3   Indicator    732 non-null    str    
 4   Value        732 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 80.0 KB
None

stability
<class 'pandas.DataFrame'>
RangeIndex: 3526 entries, 0 to 3525
Data colu

## Concaténation

In [75]:
df_facts = pd.concat(dfs.values(), ignore_index=True)
df_facts.name = "Facts"
df_facts.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 37778 entries, 0 to 37777
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      37778 non-null  str    
 1   Year         37778 non-null  int64  
 2   Granularity  37778 non-null  str    
 3   Indicator    37778 non-null  str    
 4   Value        37778 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 3.0 MB


## Nombre de pays/années par indicateur/granularité

In [76]:
df_facts.groupby(["Indicator","Granularity"])[["Country","Year"]].nunique()

Country  Year
Indicator                                          Granularity               
Mortality rate attributed to exposure to unsafe... Female           183     1
                                                   Male             183     1
                                                   Total            183     1
Political stability                                Total            200    18
Population                                         Female           204    19
                                                   Male             204    19
                                                   Rural            236    19
                                                   Total            238    19
                                                   Urban            236    19
Population using at least basic drinking-water ... Rural            166    18
                                                   Total            194    18
                                                   Urban            169    18
Population using safely managed drinking-water ... Rural             34    18
                                                   Total             98    18
                                                   Urban             52    18
WASH deaths                                        Total            183     1

# Construction des tables de dimension

## Dimension Country

In [77]:
df_dim_country = df_iso3.merge(df_region_country, how="outer", indicator="from")
df_dim_country.name = "Country"
df_dim_country

,Country,ISO3,Region,from
0,Afghanistan,AFG,Eastern Mediterranean,both
1,Albania,ALB,Europe,both
2,Algeria,DZA,Africa,both
3,American Samoa,ASM,NaN,left_only
4,Andorra,AND,Europe,both
...,...,...,...,...
233,Wallis and Futuna Islands,WLF,NaN,left_only
234,Western Sahara,ESH,NaN,left_only
235,Yemen,YEM,Eastern Mediterranean,both
236,Zambia,ZMB,Africa,both


In [78]:
df_dim_country["from"].value_counts()

from
both          197
left_only      41
right_only      0
Name: count, dtype: int64

In [79]:
del df_dim_country["from"]

## Dimension Year

In [80]:
df_dim_year = pd.DataFrame({ "Year": sorted(df_facts["Year"].unique()) })
df_dim_year.name = "Year"
df_dim_year

,Year
0,2000
1,2001
2,2002
3,2003
4,2004
5,2005
6,2006
7,2007
8,2008
9,2009


## Dimension Granularity

In [81]:
df_dim_granularity = pd.DataFrame({ "Granularity": sorted(df_facts["Granularity"].unique()) })
df_dim_granularity.name = "Granularity"
df_dim_granularity

,Granularity
0,Female
1,Male
2,Rural
3,Total
4,Urban


In [82]:
df_dim_granularity = pd.DataFrame([
    [  "Total",  True,  True ],
    [   "Male",  True, False ],
    [ "Female",  True, False ],
    [  "Urban", False,  True ],
    [  "Rural", False,  True ],
], columns=["Granularity","IsMaleFemaleGranularity","IsUrbanRuralGranularity"])
df_dim_granularity.name = "Granularity"
df_dim_granularity

,Granularity,IsMaleFemaleGranularity,IsUrbanRuralGranularity
0,Total,True,True
1,Male,True,False
2,Female,True,False
3,Urban,False,True
4,Rural,False,True


## Dimension Indicator

In [83]:
df_dim_indicator = pd.DataFrame({ "Indicator": sorted(df_facts["Indicator"].unique()) })
df_dim_indicator.name = "Indicator"
df_dim_indicator

,Indicator
0,Mortality rate attributed to exposure to unsaf...
1,Political stability
2,Population
3,Population using at least basic drinking-water...
4,Population using safely managed drinking-water...
5,WASH deaths


In [84]:
df_dim_indicator = pd.DataFrame([
    [ "Population using at least basic drinking-water services", "Ratio (raw)", "WHO" ],
    [ "Population using safely managed drinking-water services", "Ratio (raw)", "WHO" ],
    [ "Mortality rate attributed to exposure to unsafe WASH services", "Ratio per 100k", "WHO" ],
    [ "WASH deaths", "Number of people", "WHO" ],
    [ "Political stability", "Standardized value (ABS(outlier) >= 2.5)", "FAO" ],
    [ "Population", "Number of people", "FAO" ],
], columns=["Indicator","Comments","Source"])
df_dim_indicator.name = "Indicator"
df_dim_indicator

,Indicator,Comments,Source
0,Population using at least basic drinking-water...,Ratio (raw),WHO
1,Population using safely managed drinking-water...,Ratio (raw),WHO
2,Mortality rate attributed to exposure to unsaf...,Ratio per 100k,WHO
3,WASH deaths,Number of people,WHO
4,Political stability,Standardized value (ABS(outlier) >= 2.5),FAO
5,Population,Number of people,FAO


# Changement de format de la table de faits (long vers large)

In [85]:
df_facts_pivot = df_facts.pivot(index=["Country","Year"], columns=["Indicator","Granularity"], values="Value").reset_index()
df_facts_pivot.name = "Facts"
df_facts_pivot.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 4430 entries, 0 to 4429
Data columns (total 18 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   (Country, )                                                              4430 non-null   str    
 1   (Year, )                                                                 4430 non-null   int64  
 2   (Population using at least basic drinking-water services, Rural)         2953 non-null   float64
 3   (Population using at least basic drinking-water services, Total)         3449 non-null   float64
 4   (Population using at least basic drinking-water services, Urban)         3013 non-null   float64
 5   (Population using safely managed drinking-water services, Total)         1745 non-null   float64
 6   (Population using safely managed drinking-water services, Urban)         929 non-nu

In [86]:
mapping_pivot = {
    ("Country", ""): "Country",
    ("Year", ""): "Year",
    ("Population using at least basic drinking-water services", "Total"): "Basic Total",
    ("Population using at least basic drinking-water services", "Rural"): "Basic Rural",
    ("Population using at least basic drinking-water services", "Urban"): "Basic Urban",
    ("Population using safely managed drinking-water services", "Total"): "Safe Total",
    ("Population using safely managed drinking-water services", "Urban"): "Safe Urban",
    ("Population using safely managed drinking-water services", "Rural"): "Safe Rural",
    ("Mortality rate attributed to exposure to unsafe WASH services",  "Total"): "Mortality Total" ,
    ("Mortality rate attributed to exposure to unsafe WASH services",   "Male"): "Mortality Male"  ,
    ("Mortality rate attributed to exposure to unsafe WASH services", "Female"): "Mortality Female",
    ("WASH deaths", "Total"): "Deaths",
    ("Political stability", "Total"): "Political stability",
    ("Population",  "Total"): "Population Total" ,
    ("Population",   "Male"): "Population Male"  ,
    ("Population", "Female"): "Population Female",
    ("Population",  "Rural"): "Population Rural" ,
    ("Population",  "Urban"): "Population Urban" ,
}
# df_facts_pivot.rename(columns=mapping_pivot, inplace=True)
df_facts_pivot.columns = map(mapping_pivot.__getitem__, df_facts_pivot.columns)
df_facts_pivot.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 4430 entries, 0 to 4429
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              4430 non-null   str    
 1   Year                 4430 non-null   int64  
 2   Basic Rural          2953 non-null   float64
 3   Basic Total          3449 non-null   float64
 4   Basic Urban          3013 non-null   float64
 5   Safe Total           1745 non-null   float64
 6   Safe Urban           929 non-null    float64
 7   Safe Rural           612 non-null    float64
 8   Mortality Female     183 non-null    float64
 9   Mortality Male       183 non-null    float64
 10  Mortality Total      183 non-null    float64
 11  Deaths               183 non-null    float64
 12  Political stability  3526 non-null   float64
 13  Population Total     4411 non-null   float64
 14  Population Male      3809 non-null   float64
 15  Population Female    3809 non-null   float64
 16 

# Export

`pandas` a besoin de `xlsxwriter` ou `openpyxl` (engine) pour pouvoir exporter au format Excel.

In [87]:
excel_path = data_path/"model_long.xlsx"
print(f'opening file "{excel_path}"', end=' ... ', flush=True)
with pd.ExcelWriter(excel_path) as writer:
    print('done')
    print(f'engine = {writer.engine}')
    for df in (df_facts, df_dim_country, df_dim_year, df_dim_granularity, df_dim_indicator):
        print(f'creating sheet "{df.name}"', end=' ... ', flush=True)
        df.to_excel(writer, index=False, sheet_name=df.name)
        print('done')
    print(f'writing to file "{excel_path}"', end=' ... ', flush=True)
print('done')

opening file "../data/model_long.xlsx" ... done
engine = openpyxl
creating sheet "Facts" ... done
creating sheet "Country" ... done
creating sheet "Year" ... done
creating sheet "Granularity" ... done
creating sheet "Indicator" ... done
writing to file "../data/model_long.xlsx" ... done


In [88]:
excel_path = data_path/"model_wide.xlsx"
print(f'opening file "{excel_path}"', end=' ... ', flush=True)
with pd.ExcelWriter(excel_path) as writer:
    print('done')
    print(f'engine = {writer.engine}')
    for df in (df_facts_pivot, df_dim_country, df_dim_year):
        print(f'creating sheet "{df.name}"', end=' ... ', flush=True)
        df.to_excel(writer, index=False, sheet_name=df.name)
        print('done')
    print(f'writing to file "{excel_path}"', end=' ... ', flush=True)
print('done')

opening file "../data/model_wide.xlsx" ... done
engine = openpyxl
creating sheet "Facts" ... done
creating sheet "Country" ... done
creating sheet "Year" ... done
writing to file "../data/model_wide.xlsx" ... done
